In [15]:
!pip install "setuptools<71.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.1/931.1 kB 11.4 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


In [16]:
import torch
from typing import Dict

from tdc.single_pred import Develop

import graphein.protein as gp
from graphein.ml.conversion import GraphFormatConvertor
from graphein.ml import InMemoryProteinGraphDataset, ProteinGraphDataset

In [17]:
data = Develop(name = 'SAbDab_Chen')
split = data.get_split()
split["train"].head()

Downloading...


  0%|          | 0.00/601k [00:00<?, ?iB/s]

  0%|          | 1.02k/601k [00:00<06:47, 1.47kiB/s]

  3%|▎         | 17.4k/601k [00:01<00:30, 18.9kiB/s]

  6%|▌         | 34.8k/601k [00:01<00:28, 19.7kiB/s]

  9%|▉         | 54.3k/601k [00:02<00:18, 30.2kiB/s]

 10%|█         | 62.5k/601k [00:03<00:37, 14.3kiB/s]

 13%|█▎        | 78.8k/601k [00:04<00:24, 21.2kiB/s]

 14%|█▍        | 84.0k/601k [00:05<00:45, 11.4kiB/s]

 16%|█▌        | 96.3k/601k [00:06<00:35, 14.2kiB/s]

 17%|█▋        | 99.3k/601k [00:07<00:56, 8.94kiB/s]

 19%|█▉        | 115k/601k [00:09<01:00, 7.99kiB/s] 

 22%|██▏       | 132k/601k [00:11<00:52, 8.90kiB/s]

 25%|██▍       | 148k/601k [00:11<00:34, 13.2kiB/s]

 25%|██▌       | 153k/601k [00:13<00:58, 7.66kiB/s]

 28%|██▊       | 167k/601k [00:14<00:42, 10.1kiB/s]

 31%|███       | 184k/601k [00:15<00:37, 11.1kiB/s]

 34%|███▎      | 202k/601k [00:17<00:38, 10.3kiB/s]

 36%|███▋      | 218k/601k [00:18<00:31, 12.2kiB/s]

 37%|███▋      | 220k/601k 

,Antibody_ID,Antibody,Y
0,12e8,['EVQLQQSGAEVVRSGASVKLSCTASGFNIKDYYIHWVKQRPEKG...,0
1,15c8,['EVQLQQSGAELVKPGASVKLSCTASGFNIKDTYMHWVKQKPEQG...,0
2,1a0q,['EVQLQESDAELVKPGASVKISCKASGYTFTDHVIHWVKQKPEQG...,1
3,1a14,['QVQLQQSGAELVKPGASVRMSCKASGYTFTNYNMYWVKQSPGQG...,0
4,1a2y,['QVQLQESGPGLVAPSQSLSITCTVSGFSLTGYGVNWVRQPPGKG...,0


In [18]:
from graphein.protein.utils import get_obsolete_mapping

obs = get_obsolete_mapping()

train_obs = [t for t in split["train"]["Antibody_ID"] if t in obs.keys()]
valid_obs = [t for t in split["valid"]["Antibody_ID"] if t in obs.keys()]
test_obs = [t for t in split["test"]["Antibody_ID"] if t in obs.keys()]

print(train_obs)
print(valid_obs)
print(test_obs)

['1om3', '1zls', '1zlu', '1zlw', '2f5a', '3l5y', '3qot', '3rvv', '3rvw', '3rvx', '3wxw', '4nx3', '4pp2', '4x4y', '5e99', '5kmv', '5usi', '6erx']
[]
['3wxv', '1zlv', '1op3', '3qos']


In [19]:
# If you want, you can get the PDB IDs of the new structure that replaces the obsolete entry
print("Replacement PDBs: ", [obs[t] for t  in train_obs])

# However, in this instance we will simply remove the obsolete entries from the train and test sets.
split["train"] = split["train"].loc[~split["train"]["Antibody_ID"].isin(train_obs)]
split["test"] = split["test"].loc[~split["test"]["Antibody_ID"].isin(test_obs)]

Replacement PDBs:  ['6n32', '6msy', '6mub', '6mnf', '2pr4', '4ps4', '5i18', '5vpl', '5vpg', '5vph', '6ks1', '4web', '5vco', '6dn0', '5ihu', '5vzx', '6b9j', '6fxn']


In [20]:
# Convert labels to tensors
def get_label_map(split_name: str) -> Dict[str, torch.Tensor]:
    return dict(zip(split[split_name].Antibody_ID, split[split_name].Y.apply(torch.tensor)))

train_labels = get_label_map("train")
valid_labels = get_label_map("valid")
test_labels = get_label_map("test")

In [21]:
from functools import partial

config = gp.ProteinGraphConfig(
        node_metadata_functions=[gp.amino_acid_one_hot],
        edge_construction_functions=[partial(gp.add_distance_threshold, threshold=6, long_interaction_threshold=0)]
)


In [22]:
config.dict()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [functools.partial(<function add_distance_threshold at 0x7f51b5140f40>, threshold=6, long_interaction_threshold=0)],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.amino_acid_one_hot(n, d: Dict[str, Any], return_array: bool = True, allowable_set: Optional[List[str]] = None) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_config': None,
 'dssp_config': None}

In [23]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index", "amino_acid_one_hot"])

In [32]:
g = gp.construct_graph(pdb_code="1lds", config=config)

Output()

In [42]:
print(g.graph['config'])

granularity='CA' keep_hets=[] insertions=True alt_locs='max_occupancy' pdb_dir=None verbose=False exclude_waters=True deprotonate=False protein_df_processing_functions=None edge_construction_functions=[functools.partial(<function add_distance_threshold at 0x7f51b5140f40>, threshold=6, long_interaction_threshold=0)] node_metadata_functions=[<function amino_acid_one_hot at 0x7f51b51434c0>] edge_metadata_functions=None graph_metadata_functions=None get_contacts_config=None dssp_config=None


In [34]:
ac = {}

for n in g.nodes():
    ac[g.nodes[n]["residue_name"]] = g.nodes[n]['amino_acid_one_hot']
    

In [35]:
print(ac)
    

{'MET': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'ILE': array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'GLN': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]), 'ARG': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]), 'THR': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]), 'PRO': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]), 'LYS': array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'VAL': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]), 'TYR': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]), 'SER': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]), 'HIS': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'ALA': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'GLU': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'ASN': arra